# Reducer example

A minimal, self-contained walk-through of the dimension-reduction step: build a small tenor structure, hand-write a simple covariance matrix, and run one of the reducers on it. No CSVs or dashboard required.

Change `REDUCER` in the last cell to compare methods.

In [1]:
import sys
from pathlib import Path

# Make the project package importable regardless of where Jupyter was launched.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.tenor_graph import TenorGraph
from src.reducers import REGISTRY, get_reducer

## 1. Define a tenor structure

Each tenor lists its `predictors`. Layers are inferred from the predictor chain: a tenor with no predictors is layer 0 (a liquid anchor), otherwise it sits one layer below its predictors. Here `2Y / 5Y / 10Y` are anchors and `3Y / 7Y` are derived from their bracketing anchors.

In [2]:
structure = {
    "2Y":  {"predictors": []},
    "5Y":  {"predictors": []},
    "10Y": {"predictors": []},
    "3Y":  {"predictors": ["2Y", "5Y"]},
    "7Y":  {"predictors": ["5Y", "10Y"]},
}

tg = TenorGraph(structure)
ok, errors = tg.validate()
print(tg.visualize())
print("\nvalid:", ok, errors or "")

Layer 0:
  2Y
  5Y
  10Y
Layer 1:
  3Y <- 2Y, 5Y
  7Y <- 5Y, 10Y

valid: True 


## 2. Define an example covariance matrix

`C` is the covariance of daily par-rate changes, with rows/cols aligned to `tenors` (sorted by maturity). To keep it simple but valid, build it from per-tenor volatilities and AR(1) correlations that decay as tenors move apart — this is guaranteed symmetric positive semi-definite, so every eigenvalue is positive.

In [3]:
tenors = ["2Y", "3Y", "5Y", "7Y", "10Y"]

vols = np.array([0.6, 0.7, 0.9, 1.0, 1.1])   # per-tenor daily vol
rho  = 0.9                                   # correlation decay between adjacent tenors
k    = np.arange(len(tenors))
C    = np.outer(vols, vols) * rho ** np.abs(k[:, None] - k[None, :])

print("eigenvalues:", np.linalg.eigvalsh(C).round(4))   # all > 0 => valid covariance
pd.DataFrame(C, index=tenors, columns=tenors).round(3)

eigenvalues: [0.0327 0.0613 0.1084 0.3094 3.3583]


,2Y,3Y,5Y,7Y,10Y
2Y,0.360,0.378,0.437,0.437,0.433
3Y,0.378,0.490,0.567,0.567,0.561
5Y,0.437,0.567,0.810,0.810,0.802
7Y,0.437,0.567,0.810,1.000,0.990
10Y,0.433,0.561,0.802,0.990,1.210


## 3. Run a reducer

`fit` returns a `FactorResult` whose `loadings` matrix maps tenors (rows) onto factors (columns). Each factor is labelled by its anchor tenor. Switch `REDUCER` to any key in the registry to compare methods.

In [4]:
print("available reducers:", list(REGISTRY))

REDUCER = "sequential_pca"   # try: sequential_ols, local_pca, local_ols

result = get_reducer(REDUCER).fit(C, tenors, tg)

loadings = pd.DataFrame(result.loadings, index=result.tenors, columns=result.factor_labels)
loadings.round(3)

available reducers: ['sequential_pca', 'sequential_ols', 'local_pca', 'local_ols']


,2Y,5Y,10Y,3Y,7Y
2Y,1.000,-0.000,0.000,0.0,0.0
3Y,0.767,0.316,-0.021,1.0,-0.0
5Y,-0.000,1.000,0.000,0.0,0.0
7Y,-0.564,1.027,0.358,-0.0,1.0
10Y,-0.000,0.000,1.000,0.0,0.0
